# Sieci neuronowe i deep learning - Lab 1

## Wprowadzenie do deep learningu i ewaluacji modeli

W tym laboratorium budujemy pierwszy prosty eksperyment uczenia nadzorowanego. Część związana z danymi, podziałem, skalowaniem, baseline'ami i metrykami korzysta ze `scikit-learn`, ponieważ te elementy są wspólne dla wielu frameworków deep learningu.

Wersja modelu: **PyTorch**. Notebook jest tłumaczeniem wersji TensorFlow/Keras na PyTorch.

## Ćwiczenie 1

- Załaduj zbiór danych `breast_cancer`.
- Sprawdź liczbę przykładów, liczbę cech, nazwy klas i liczności klas.
- Obejrzyj kilka wybranych cech. Zastanów się, czy dane wymagają skalowania.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

In [ ]:
data = load_breast_cancer()

X = data.data.astype("float32")
y = data.target.astype("int64")

print("Liczba przykładów:", X.shape[0])
print("Liczba cech:", X.shape[1])
print("Nazwy klas:", dict(enumerate(data.target_names)))
print("Liczności klas:", dict(zip(*np.unique(y, return_counts=True))))

In [ ]:
# Uzupełnij tę komórkę o własne sprawdzenia danych.
# Przykłady:
# - kształt macierzy X,
# - zakres wartości cech,
# - pierwsze etykiety,
# - kilka nazw cech.

# TODO: wpisz własny kod poniżej

In [ ]:
# Wybierz dwie cechy i porównaj klasy na wykresie.
# Zmień indeksy cech i sprawdź, czy klasy są łatwe do rozdzielenia.

feature_x = 0
feature_y = 1

# TODO: narysuj wykres punktowy plt.scatter(...)
# TODO: podpisz osie nazwami cech
# TODO: wyświetl wykres

## Ćwiczenie 2

- Podziel dane na zbiory treningowy, walidacyjny i testowy.
- Zachowaj proporcje klas przez parametr `stratify`.
- Dopasuj `StandardScaler` wyłącznie na zbiorze treningowym.

In [ ]:
# Uzupełnij parametry podziału.
# Zbiór testowy powinien zostać odłożony do końcowej oceny.

test_size = ...
validation_size = ...

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=test_size,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_raw,
    y_train,
    test_size=validation_size,
    stratify=y_train,
    random_state=RANDOM_STATE,
)

print("train:", X_train_raw.shape, np.bincount(y_train))
print("validation:", X_val_raw.shape, np.bincount(y_val))
print("test:", X_test_raw.shape, np.bincount(y_test))

In [ ]:
# Dopasuj StandardScaler tylko na zbiorze treningowym.
# Następnie przeskaluj walidację i test tym samym skalerem.

scaler = StandardScaler()

X_train = ...
X_val = ...
X_test = ...

print("Średnia 5 pierwszych cech w train:", X_train[:, :5].mean(axis=0).round(3))
print("Odchylenie 5 pierwszych cech w train:", X_train[:, :5].std(axis=0).round(3))

## Cwiczenie 3

- Przygotuj funkcje do liczenia metryk klasyfikacji.
- Porownaj co najmniej dwa baseline'y: najczestsza klase i regresje logistyczna.
- Wyswietl macierz pomylek dla jednego z modeli.
- W `classification_report` ustaw `zero_division=0`. Baseline przewidujacy jedna klase nie ma zdefiniowanej precision dla klas bez predykcji.

In [ ]:
def evaluate_predictions(name, y_true, y_pred, y_score=None):
    print(f"## {name}")

    # TODO: policz accuracy
    # TODO: policz balanced accuracy
    # TODO: jezeli y_score nie jest None, policz ROC AUC
    # TODO: wyswietl classification_report z zero_division=0
    pass

In [ ]:
# Baseline 1: model przewidujący najczęstszą klasę.

dummy = DummyClassifier(strategy="most_frequent")
# TODO: wytrenuj dummy na zbiorze treningowym
# TODO: oblicz predykcje na walidacji
# TODO: uruchom evaluate_predictions


# Baseline 2: klasyczny model liniowy.

log_reg = LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
# TODO: wytrenuj regresję logistyczną
# TODO: oblicz klasy i score/probability na walidacji
# TODO: porównaj wynik z baseline'em najczęstszej klasy

In [ ]:
# TODO: wyświetl macierz pomyłek dla regresji logistycznej albo innego wybranego modelu.
# Podpowiedź: użyj ConfusionMatrixDisplay.from_predictions(...).

## Ćwiczenie 4

- Stwórz prostą sieć MLP w PyTorch.
- Uzupełnij lub zmień architekturę: liczbę warstw, liczbę neuronów i funkcje aktywacji.
- Zwróć uwagę, że `BCEWithLogitsLoss` oczekuje logitów, więc ostatnia warstwa nie ma sigmoidu.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(RANDOM_STATE)


def make_loader(X_array, y_array, batch_size=32, shuffle=False):
    X_tensor = torch.tensor(X_array, dtype=torch.float32)
    y_tensor = torch.tensor(y_array.reshape(-1, 1), dtype=torch.float32)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=shuffle)


train_loader = make_loader(X_train, y_train, batch_size=32, shuffle=True)
val_loader = make_loader(X_val, y_val, batch_size=128, shuffle=False)

model = nn.Sequential(
    # TODO: dodaj warstwę Linear
    # TODO: dodaj aktywację ReLU
    # TODO: dodaj drugą warstwę ukrytą albo zmień architekturę
    # TODO: dodaj warstwę wyjściową z jednym logitem
)

model

## Ćwiczenie 5

- Wytrenuj model.
- Narysuj krzywe uczenia dla funkcji straty i dokładności.
- Porównaj wynik na treningu i walidacji.

In [ ]:
loss_fn = ...       # TODO: wybierz funkcję straty dla klasyfikacji binarnej z logitami
optimizer = ...     # TODO: wybierz optymalizator


def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:
        # TODO: wyzeruj gradienty w trybie treningowym
        # TODO: policz logity
        # TODO: policz stratę
        # TODO: wykonaj backward i optimizer.step w trybie treningowym
        # TODO: policz predykcje po sigmoidzie
        pass

    return total_loss / total, correct / total


history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

# TODO: napisz pętlę po epokach i zapisuj wyniki treningu oraz walidacji

In [ ]:
# TODO: narysuj krzywe uczenia dla loss i val_loss.
# TODO: narysuj krzywe accuracy i val_accuracy.
# Podpowiedź: historia treningu jest w history.history.

In [ ]:
def predict_torch(model, X_eval):
    # TODO: przełącz model w tryb eval
    # TODO: zamień X_eval na tensor
    # TODO: policz logity bez gradientów
    # TODO: zamień logity na prawdopodobieństwa przez sigmoid
    # TODO: zamień prawdopodobieństwa na klasy progiem 0.5
    pass


# TODO: oceń model na zbiorze walidacyjnym

## Cwiczenie 6

- Zbuduj kontrolowany przyklad przeuczenia.
- Wez bardzo maly podzbior treningowy i celowo zastap jego etykiety losowymi etykietami.
- Wytrenuj duzy model bez regularyzacji tak, aby mogl zapamietac te przyklady.
- Porownaj accuracy/loss na malym zbiorze treningowym oraz na normalnym zbiorze walidacyjnym.

To nie jest poprawna procedura uczenia. To eksperyment pokazujacy, ze duzy model moze nauczyc sie szumu na train, ale nie uogolni tego na validation.

In [ ]:
# TODO: wybierz np. 40 losowych przykladow ze zbioru treningowego.
# TODO: utworz dla nich losowe etykiety 0/1. Nie uzywaj prawdziwych y_small.
# TODO: zbuduj duzy model PyTorch bez dropout/regularyzacji.
# TODO: trenuj go na losowych etykietach, ale waliduj na normalnym val_loader.
# TODO: narysuj train/validation loss oraz accuracy.
#
# Podpowiedz: to cwiczenie ma pokazac memorization. Jezeli train accuracy rosnie,
# a validation accuracy/loss sa slabe, przyklad dziala poprawnie.

## Ćwiczenie 7

- Uruchom sztuczny przykład data leakage.
- Wyjaśnij, dlaczego wynik takiego modelu nie może być traktowany jako prawdziwa ocena jakości.

In [ ]:
# Sztuczny przykład data leakage:
# do cech dopisujemy etykietę klasy, czyli informację, której model nie powinien znać.

X_with_leak = np.column_stack([X, y]).astype("float32")

# TODO: podziel X_with_leak na train/test
# TODO: wytrenuj prosty model
# TODO: porównaj wynik z normalną ewaluacją
# TODO: wyjaśnij, dlaczego ten wynik nie jest wiarygodny

## Ćwiczenie 8

- Wybierz model na podstawie wyników walidacyjnych.
- Dopiero teraz użyj zbioru testowego.
- Zapisz końcowe metryki i macierz pomyłek.

In [ ]:
# TODO: po wybraniu modelu na podstawie walidacji użyj zbioru testowego.
# TODO: policz końcowe metryki.
# TODO: wyświetl macierz pomyłek.